# Kaggle Pipeline (Raw Extraction Only)
Executes the pipeline WITHOUT denoising, normalization, and augmentations. Only original audio features are extracted.

In [ ]:
!pip install librosa soundfile xgboost scikit-learn pandas numpy tqdm joblib

import os
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
import warnings


In [ ]:
# UPDATE THIS PATH to match your actual Kaggle dataset input path
DATASET_PATH = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/data"
OUTPUT_DIR = "/kaggle/working/"

In [ ]:
def extract_advanced_features(y, sr):
    features = {}
    try:
        rms = librosa.feature.rms(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y=y)[0]
        centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
        bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        
        def add_stats(name, array):
            features[f'{name}_mean'] = np.mean(array)
            features[f'{name}_std'] = np.std(array)
            features[f'{name}_max'] = np.max(array)
            features[f'{name}_min'] = np.min(array)
            
        add_stats('rms', rms)
        add_stats('zcr', zcr)
        add_stats('centroid', centroid)
        add_stats('rolloff', rolloff)
        add_stats('bandwidth', bandwidth)
        
        for i in range(13):
            add_stats(f'mfcc_{i+1}', mfccs[i])
        for i in range(contrast.shape[0]):
            add_stats(f'contrast_{i+1}', contrast[i])
            
        return features
    except Exception as e:
        return {}

In [ ]:
def process_file(file_path):
    results = []
    try:
        y, sr = librosa.load(file_path, sr=None)
        machine_name = file_path.parent.parent.parent.name
        label = file_path.parent.name
        if not machine_name.startswith("Machine"):
            return []
            
        # No denoising, no normalization, no augmentation
        features = extract_advanced_features(y, sr)
        if features:
            features['file_path'] = str(file_path)
            features['machine_name'] = machine_name
            features['label'] = label
            features['is_augmented'] = False
            results.append(features)
    except Exception as e:
        pass
    return results

root_dir = Path(DATASET_PATH)
wav_files = list(root_dir.rglob("*.wav"))
print(f"Found {len(wav_files)} files. Starting extraction...")

# Using joblib (Loky backend)
all_results = Parallel(n_jobs=-1, batch_size="auto")(
    delayed(process_file)(f) for f in tqdm(wav_files, desc="Extracting Features")
)

flat_results = [item for sublist in all_results for item in sublist if item]

df = pd.DataFrame(flat_results)
df['fused_label'] = df['machine_name'] + '_' + df['label']
df.to_csv(os.path.join(OUTPUT_DIR, 'kaggle_extracted_features_raw.csv'), index=False)



In [ ]:
# 1. Feature Pruning
exclude_cols = ['file_path', 'machine_name', 'label', 'fused_label', 'is_augmented']
feature_cols = [c for c in df.columns if c not in exclude_cols]

X_full = df[feature_cols].fillna(0)
y_full = df['fused_label']

print("Running Random Forest to prune features...")
rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_full, y_full)
importances = pd.DataFrame({'Feature': feature_cols, 'Importance': rf.feature_importances_})
top_40_features = importances.sort_values(by='Importance', ascending=False)['Feature'].head(40).tolist()
X_pruned = df[top_40_features].fillna(0)

# 2. Intra-Class Clustering for Leakage Prevention
print("Clustering within classes for strict grouping...")
df['leakage_group_id'] = -1
group_id_counter = 0

scaler_cluster = StandardScaler()
X_scaled_for_clustering = scaler_cluster.fit_transform(X_pruned)

for cls in df['fused_label'].unique():
    cls_mask = df['fused_label'] == cls
    X_cls = X_scaled_for_clustering[cls_mask]
    
    n_clusters = min(10, len(X_cls)) 
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cls_clusters = kmeans.fit_predict(X_cls)
    
    df.loc[cls_mask, 'leakage_group_id'] = cls_clusters + group_id_counter
    group_id_counter += n_clusters

print(f"Total unique acoustic clusters formed: {group_id_counter}")

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_full)
groups = df['leakage_group_id']

# 1. Global 80/20 Train-Test Split (Strictly Grouped)
print("Splitting data into 80% Train and 20% Unseen Test sets...")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_pruned, y_encoded, groups=groups))

X_train = X_pruned.iloc[train_idx]
y_train = y_encoded[train_idx]
groups_train = groups.iloc[train_idx]

X_test = X_pruned.iloc[test_idx]
y_test = y_encoded[test_idx]

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. 5-Fold Grouped Cross Validation on Training Data
print("\nRunning 5-Fold GroupKFold Cross Validation on the Training Set...")
gkf = GroupKFold(n_splits=5)
xgb = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', random_state=42, tree_method='hist', device='cuda') # Using GPU for faster training

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1]
}

grid_search = GridSearchCV(xgb, param_grid, cv=gkf, scoring='f1_macro', n_jobs=-1, verbose=1)
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')
grid_search.fit(X_train_scaled, y_train, groups=groups_train)

best_xgb = grid_search.best_estimator_
print(f"\nBest XGBoost Parameters from CV: {grid_search.best_params_}")
print(f"Best CV Macro F1-Score: {grid_search.best_score_:.4f}")

# 3. Final Evaluation on Unseen Test Data
print("\n--- FINAL EVALUATION ON UNSEEN HOLD-OUT TEST SET ---")
preds = best_xgb.predict(X_test_scaled)
macro_f1 = f1_score(y_test, preds, average='macro')
print(f"Final Test Set Macro F1: {macro_f1:.4f}\n")

target_names = [str(cls) for cls in le.classes_]
print(classification_report(y_test, preds, target_names=target_names))